# MentalChat16K: corrected linguistic analysis (v3)

**What this notebook changes relative to NewWork_MC16k_v2_5**

1. **Row accounting**: starts from the *raw* CSV and logs every exclusion, so N is reproducible.
2. **Tokenisation fixes**: contractions are kept intact (`don't`, `you're`), so negation (`n't`) and
   pronouns (`you're`, `I've`) are no longer missed.
3. **Length-robust features**: counts become rates per 100 words; TTR gets a length-robust companion (MATTR).
4. **Hedging**: a strict lexicon, an "ambiguous words" lexicon (`can`, `kind`, `type`...), and your original list
   are reported separately, plus a word-level breakdown of what drives the gap.
5. **Third person** no longer includes `it`/`its`. **SMOG** is flagged when a response has < 30 sentences.
6. **Stats**: Welch + Mann-Whitney, Cohen's d + Cliff's delta, Holm correction, and *length-matched* effect sizes.
7. **Classification**: grouped, stratified, repeated CV (no input leakage across folds), length-only baselines,
   VIF pruning, bootstrap CIs and permutation importance, per-class error *rates*, and the triage table.

Everything is saved to `results_v3/` so the paper's numbers can be regenerated from one run.

In [ ]:
%pip install textblob textstat pandas seaborn matplotlib scipy numpy scikit-learn

In [ ]:
import os, re, warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import textstat
from textblob import TextBlob
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
DATA_PATH = os.environ.get("MC16K_CSV", "mentalchat16k.csv")  # RAW file: instruction,input,output,source
OUT_DIR = "results_v3"
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

DROP_EXACT_DUPLICATE_OUTPUTS = False  # set True to see whether duplicates matter
EXCLUDE_FLESCH_OUTLIERS = True        # matches the paper (3 rows); logged in the accounting table
CLASS_WEIGHT = "balanced"             # set to None to reproduce the unweighted v2 setup
N_REPEATS = 3                         # repeats of 5-fold CV
PALETTE = {"real": "#4C72B0", "synthetic": "#DD8452"}

## 1. Load raw data and account for every row
The MentalChat16K paper's text reports **6,338 interview + 9,775 synthetic = 16,113** rows. (Its Table 1 has the
two row labels swapped, but the text and the average word counts make the assignment clear.)
Your analysis used N = 16,081, so the table below must explain the difference.

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding="utf-8")
assert {"input", "output", "source"} <= set(raw.columns), "Need input, output, source columns"

acct = []
def log(step, d):
    vc = d["source"].value_counts()
    acct.append({"step": step, "total": len(d),
                 "real": int(vc.get("real", 0)), "synthetic": int(vc.get("synthetic", 0))})

log("raw file", raw)
df = raw[raw["output"].fillna("").str.strip() != ""].copy()
log("drop empty/missing output", df)

dup_out = df.duplicated("output", keep="first")
print("Exact duplicate outputs (kept unless DROP_EXACT_DUPLICATE_OUTPUTS):")
_dups = df.loc[dup_out, "source"].value_counts()
print(_dups.to_string() if len(_dups) else "none", "\n")
if DROP_EXACT_DUPLICATE_OUTPUTS:
    df = df[~dup_out].copy()
    log("drop exact duplicate outputs", df)
df = df.reset_index(drop=True)

# group ids for leakage-safe CV: rows that share the same input text must stay in the same fold
_inp = df["input"].fillna("").str.strip().str.lower()
group_keys = [g if g else f"__row{i}" for i, g in enumerate(_inp)]
df["group"] = pd.factorize(pd.Series(group_keys))[0]

shared = set(df.loc[df.source == "real", "group"]) & set(df.loc[df.source == "synthetic", "group"])
print(f"Input texts that appear in BOTH classes: {len(shared)}")
print(f"Rows whose input is repeated within the dataset: {int(df.duplicated('group', keep=False).sum())}\n")

## 2. Feature extraction (fixed)

In [ ]:
APOS = str.maketrans({"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"'})
TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)*")        # keeps contractions: don't, you're, i've
SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+|\n+")

NEGATION = {"not", "no", "never", "none", "nobody", "nothing", "nowhere", "neither", "nor", "cannot"}
# plus any token ending in n't (don't, can't, won't ...), handled in code

# Strict hedge lexicon: epistemic adverbs/verbs and epistemic modals. Swap in a published lexicon if your
# mentor prefers one; this is a transparent, conservative starting point.
HEDGE_STRICT = {
    "perhaps", "maybe", "might", "could", "may", "somewhat", "probably", "likely", "possibly",
    "seems", "seem", "seemed", "appears", "appear", "suggests", "suggest", "indicates", "indicate",
    "tends", "tend", "generally", "usually", "sometimes", "approximately", "roughly", "fairly",
    "rather", "quite", "apparently", "presumably", "potentially", "arguably",
}
# Words from your v2 list that are usually NOT hedges in counselling text
HEDGE_AMBIGUOUS = {"can", "kind", "type", "sort", "certain", "around", "often"}
HEDGE_ORIGINAL = HEDGE_STRICT | HEDGE_AMBIGUOUS | {"uncertain"}  # approximates the v2 list for comparison

FIRST = {"i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves"}
SECOND = {"you", "your", "yours", "yourself", "yourselves"}
THIRD = {"he", "she", "they", "him", "her", "them", "his", "hers", "their", "theirs",
         "himself", "herself", "themselves"}  # 'it'/'its' removed: not person reference


def mattr(tokens, window=50, stride=5):
    """Moving-average type-token ratio: far less length-dependent than plain TTR."""
    n = len(tokens)
    if n == 0:
        return np.nan
    if n < window:
        return len(set(tokens)) / n
    return float(np.mean([len(set(tokens[i:i + window])) / window
                          for i in range(0, n - window + 1, stride)]))


def compute_features(text):
    t = text.translate(APOS)
    toks = TOKEN_RE.findall(t.lower())
    n = len(toks)
    sents = [s for s in SENT_SPLIT_RE.split(t) if TOKEN_RE.search(s.lower())]
    n_sent = max(len(sents), 1)
    base = [w.split("'")[0] for w in toks]            # "you're" -> "you", "i've" -> "i"
    per100 = 100.0 / max(n, 1)

    f = {"word_count": n, "sentence_count": n_sent, "char_count": len(t),
         "avg_words_per_sentence": n / n_sent,
         "avg_chars_per_word": sum(len(w.replace("'", "")) for w in toks) / max(n, 1),
         "ttr": len(set(toks)) / max(n, 1), "mattr": mattr(toks)}

    # readability (SMOG assumes ~30 sentences, so it is flagged rather than trusted)
    f["flesch_reading_ease"] = textstat.flesch_reading_ease(t)
    f["gunning_fog"] = textstat.gunning_fog(t)
    f["smog"] = textstat.smog_index(t)
    f["smog_valid"] = int(n_sent >= 30)
    f["dale_chall"] = textstat.dale_chall_readability_score(t)

    blob = TextBlob(t)
    f["polarity"] = blob.sentiment.polarity
    f["subjectivity"] = blob.sentiment.subjectivity

    f["negation_count"] = sum((w in NEGATION) or w.endswith("n't") for w in toks)
    f["hedge_count_strict"] = sum(w in HEDGE_STRICT for w in toks)
    f["hedge_count_ambiguous"] = sum(w in HEDGE_AMBIGUOUS for w in toks)
    f["hedge_count_original"] = sum(w in HEDGE_ORIGINAL for w in toks)
    f["first_person_count"] = sum(w in FIRST for w in base)
    f["second_person_count"] = sum(w in SECOND for w in base)
    f["third_person_count"] = sum(w in THIRD for w in base)
    f["question_mark_count"] = t.count("?")
    f["ellipsis_count"] = t.count("...") + t.count("\u2026")

    rate_map = {"negation_rate": "negation_count", "hedge_strict_rate": "hedge_count_strict",
                "hedge_ambiguous_rate": "hedge_count_ambiguous", "hedge_original_rate": "hedge_count_original",
                "first_person_rate": "first_person_count", "second_person_rate": "second_person_count",
                "third_person_rate": "third_person_count"}
    for rate, cnt in rate_map.items():
        f[rate] = f[cnt] * per100
    f["question_rate"] = f["question_mark_count"] * per100
    f["ellipsis_rate"] = f["ellipsis_count"] * per100
    return f


CACHE = os.path.join(OUT_DIR, "features_v3.csv")
FORCE_RECOMPUTE = False  # set True (or delete the cache file) after editing compute_features()
if (not FORCE_RECOMPUTE) and os.path.exists(CACHE) and len(pd.read_csv(CACHE, usecols=[0])) == len(df):
    feats = pd.read_csv(CACHE)
    print("Loaded cached features:", CACHE)
else:
    print("Computing features (a few minutes for ~16k texts)...")
    feats = pd.DataFrame([compute_features(x) for x in df["output"]])
    feats.to_csv(CACHE, index=False)
df = pd.concat([df.reset_index(drop=True), feats], axis=1)

# Flesch outliers: flag, log, optionally exclude
df["flesch_outlier"] = (df["flesch_reading_ease"] < 0).astype(int)
print("Flesch < 0 by source:", df.loc[df.flesch_outlier == 1, "source"].value_counts().to_dict())
if EXCLUDE_FLESCH_OUTLIERS:
    df = df[df["flesch_outlier"] == 0].reset_index(drop=True)
    log("exclude Flesch < 0", df)

acct_df = pd.DataFrame(acct)
acct_df.to_csv(os.path.join(OUT_DIR, "row_accounting.csv"), index=False)
print("\nROW ACCOUNTING (paste into the Methods section):")
print(acct_df.to_string(index=False))
print("\nPaper-reported raw size: 6,338 real + 9,775 synthetic = 16,113")

## 3. Feature sets
* `LENGTH`: raw size only.
* `STYLE`: everything else, with counts converted to **rates per 100 words**.
* `CONTENT_PRAGMATIC`: sentiment + rate-based pragmatic features + MATTR (no sentence-structure or readability
  features), to test whether any signal remains beyond sentence structure.

In [ ]:
LENGTH = ["word_count", "sentence_count"]
STYLE = ["avg_words_per_sentence", "avg_chars_per_word", "mattr",
         "flesch_reading_ease", "gunning_fog", "dale_chall",
         "polarity", "subjectivity",
         "negation_rate", "hedge_strict_rate", "first_person_rate", "second_person_rate",
         "third_person_rate", "question_rate"]
ALL = LENGTH + STYLE
CONTENT_PRAGMATIC = ["mattr", "polarity", "subjectivity", "negation_rate", "hedge_strict_rate",
                     "first_person_rate", "second_person_rate", "third_person_rate", "question_rate"]

# raw-count features shown only to make the length confound visible
COUNT_FEATURES = ["hedge_count_original", "negation_count", "first_person_count",
                  "second_person_count", "third_person_count", "question_mark_count", "ellipsis_count"]
STAT_FEATURES = list(dict.fromkeys(
    LENGTH + STYLE + ["ttr", "smog", "hedge_original_rate", "hedge_ambiguous_rate", "ellipsis_rate"] + COUNT_FEATURES))

real = df[df.source == "real"]
syn = df[df.source == "synthetic"]
print(f"Analytic sample: N = {len(df)}  (real {len(real)}, {len(real)/len(df):.1%}; "
      f"synthetic {len(syn)}, {len(syn)/len(df):.1%})")

## 4. Group comparisons
Welch t and Mann-Whitney U (counts are skewed), Cohen's d and Cliff's delta, Holm-adjusted p-values over all
tests, and a **length-matched** d: the effect size computed *within* 50-word bins and averaged, which shows how
much of each difference survives once length is held roughly constant.

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan

def cliffs_delta(a, b):
    try:
        u = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
        return 2 * u / (len(a) * len(b)) - 1
    except ValueError:
        return np.nan

def holm_adjust(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty(len(p))
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (len(p) - rank) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def effect_label(d):
    d = abs(d)
    return "negligible" if d < 0.2 else "small" if d < 0.5 else "medium" if d < 0.8 else "large"

BINS = list(range(0, 901, 50)) + [np.inf]
df["len_bin"] = pd.cut(df["word_count"], bins=BINS)

def within_bin_d(feature, min_n=30):
    ds, ws, used = [], [], 0
    for _, g in df.groupby("len_bin", observed=True):
        a = g.loc[g.source == "real", feature].values
        b = g.loc[g.source == "synthetic", feature].values
        if len(a) >= min_n and len(b) >= min_n:
            ds.append(cohens_d(a, b))
            ws.append(len(a) * len(b) / (len(a) + len(b)))
            used += len(a) + len(b)
    return (np.average(ds, weights=ws), used / len(df)) if ds else (np.nan, 0.0)

rows = []
for feat in STAT_FEATURES:
    a, b = real[feat].dropna().values, syn[feat].dropna().values
    t, p_t = stats.ttest_ind(a, b, equal_var=False)
    try:
        p_u = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
    except ValueError:
        p_u = np.nan
    d = cohens_d(a, b)
    d_lm, cov = within_bin_d(feat) if feat != "word_count" else (np.nan, np.nan)
    rows.append({"feature": feat, "mean_real": a.mean(), "mean_synth": b.mean(), "welch_t": t, "p_welch": p_t,
                 "p_mwu": p_u, "cohens_d": d, "effect": effect_label(d) if not np.isnan(d) else "n/a",
                 "cliffs_delta": cliffs_delta(a, b), "d_length_matched": d_lm, "length_bin_coverage": cov})
res = pd.DataFrame(rows)
res["p_welch_holm"] = holm_adjust(res["p_welch"].fillna(1))
res["p_welch_fmt"] = res["p_welch_holm"].map(fmt_p)
res = res.sort_values("cohens_d", key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
res.to_csv(os.path.join(OUT_DIR, "group_comparison.csv"), index=False)

show = ["feature", "mean_real", "mean_synth", "cohens_d", "effect", "cliffs_delta",
        "d_length_matched", "p_welch_fmt"]
print(res[show].round(3).to_string(index=False))

n_large = int((res["cohens_d"].abs() >= 0.8).sum())
print(f"\nFeatures with |d| >= 0.8 (Cohen 'large'): {n_large}")

In [ ]:
# Figure: overall d vs length-matched d for the features that matter
plot_feats = [f for f in (LENGTH + STYLE + ["hedge_original_rate"]) if f != "word_count"]
pf = res[res.feature.isin(plot_feats)].copy()
pf = pf.reindex(pf["cohens_d"].abs().sort_values().index)
y_pos = np.arange(len(pf))
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(y_pos - 0.2, pf["cohens_d"], height=0.4, label="Overall", color="#4C72B0")
ax.barh(y_pos + 0.2, pf["d_length_matched"], height=0.4, label="Length-matched (within 50-word bins)", color="#DD8452")
ax.set_yticks(y_pos); ax.set_yticklabels(pf["feature"])
for x in (-0.8, -0.5, -0.2, 0.2, 0.5, 0.8):
    ax.axvline(x, color="gray", lw=0.5, ls=":")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Cohen's d  (positive = higher in real)")
ax.set_title("Effect sizes before and after holding response length roughly constant")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "cohens_d_overall_vs_length_matched.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. What is driving the "hedging" gap?
Your v2 hedge list included `can`, `kind`, `type`, `sort`, `around`, `often`. This shows per-word rates and each
word's share of the total gap, so you can see whether "hedging" is really `can`/`may`.

In [ ]:
def word_rates(texts, vocab):
    c, total = Counter(), 0
    for t in texts:
        toks = TOKEN_RE.findall(t.translate(APOS).lower())
        total += len(toks)
        c.update(w for w in toks if w in vocab)
    return {w: 100 * c[w] / max(total, 1) for w in vocab}

hw = pd.DataFrame({"real_per100": word_rates(real["output"], HEDGE_ORIGINAL),
                   "synth_per100": word_rates(syn["output"], HEDGE_ORIGINAL)})
hw["diff"] = hw["synth_per100"] - hw["real_per100"]
gap = hw["diff"].sum()
hw["share_of_gap"] = hw["diff"] / gap if gap != 0 else np.nan
hw["in_strict_list"] = hw.index.isin(HEDGE_STRICT)
hw = hw.sort_values("diff", key=abs, ascending=False)
hw.to_csv(os.path.join(OUT_DIR, "hedge_word_breakdown.csv"))
print(hw.head(15).round(3).to_string())
print(f"\nShare of the total gap from AMBIGUOUS words: {hw.loc[~hw.in_strict_list, 'diff'].sum() / gap:.1%}"
      if gap != 0 else "")

## 6. Kurtosis (with the zero-variance case handled)

In [ ]:
kur = []
for feat in STAT_FEATURES:
    row = {"feature": feat}
    for name, g in (("real", real), ("synthetic", syn)):
        v = g[feat].dropna().values
        row[name] = stats.kurtosis(v, fisher=True) if np.std(v) > 0 else np.nan  # undefined if constant
    kur.append(row)
kur = pd.DataFrame(kur)
kur.to_csv(os.path.join(OUT_DIR, "kurtosis.csv"), index=False)
print(kur.round(2).to_string(index=False))
print("\nNaN = zero variance in that group (kurtosis undefined), e.g. real ellipsis_count.")

## 7. Classification with leakage-safe, repeated CV
* `StratifiedGroupKFold` keeps rows with the same input together.
* Scaling lives inside a `Pipeline`, so it is fit only on training folds.
* Several feature sets, so you can see how much of the 96-97% is just length.

In [ ]:
y = (df["source"] == "real").astype(int).values
groups = df["group"].values

def make_lr():
    return Pipeline([("sc", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=3000, class_weight=CLASS_WEIGHT))])

def make_dt():
    return DecisionTreeClassifier(max_depth=4, class_weight=CLASS_WEIGHT, random_state=SEED)

def cv_eval(make_model, cols, repeats=N_REPEATS):
    X = df[cols].values
    recs, oof = [], np.zeros(len(df))
    for r in range(repeats):
        skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED + r)
        for tr, te in skf.split(X, y, groups):
            m = make_model().fit(X[tr], y[tr])
            p = m.predict_proba(X[te])[:, 1]
            pred = (p >= 0.5).astype(int)
            recs.append({"acc": accuracy_score(y[te], pred),
                         "f1_real": f1_score(y[te], pred, pos_label=1),
                         "f1_synth": f1_score(y[te], pred, pos_label=0),
                         "auc": roc_auc_score(y[te], p)})
            if r == 0:
                oof[te] = p
    return pd.DataFrame(recs), oof

FEATURE_SETS = {
    "sentence_count only": ["sentence_count"],
    "length only (words + sentences)": LENGTH,
    "content/pragmatic only (no structure)": CONTENT_PRAGMATIC,
    "style only (no raw length)": STYLE,
    "all features": ALL,
}

summary, oofs = [], {}
for set_name, cols in FEATURE_SETS.items():
    for model_name, mk in (("LogReg", make_lr), ("Tree(d=4)", make_dt)):
        rec, oof = cv_eval(mk, cols)
        oofs[(set_name, model_name)] = oof
        summary.append({"feature_set": set_name, "model": model_name, "n_features": len(cols),
                        **{f"{k}_mean": rec[k].mean() for k in rec},
                        **{f"{k}_sd": rec[k].std() for k in rec}})
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(OUT_DIR, "cv_summary.csv"), index=False)
print(summary[["feature_set", "model", "n_features", "acc_mean", "acc_sd", "f1_real_mean",
               "f1_synth_mean", "auc_mean"]].round(4).to_string(index=False))
print("\nsd is across 5 folds x", N_REPEATS, "repeats (folds overlap, so treat it as indicative, not a formal CI).")

## 8. Multicollinearity: VIF pruning, bootstrap coefficients, permutation importance
Your v2 Logit did not converge and showed sign flips (word_count positive, sentence_count strongly negative), the
usual symptom of collinear predictors. Here we prune by VIF and re-estimate with bootstrap intervals.

In [ ]:
def vif_series(X):
    Z = ((X - X.mean()) / X.std(ddof=0)).values
    out = {}
    for i, c in enumerate(X.columns):
        yy = Z[:, i]
        Xo = np.column_stack([np.ones(len(Z)), np.delete(Z, i, axis=1)])
        beta, *_ = np.linalg.lstsq(Xo, yy, rcond=None)
        r2 = 1 - (yy - Xo @ beta).var() / yy.var()
        out[c] = 1 / max(1 - r2, 1e-9)
    return pd.Series(out).sort_values(ascending=False)

PROTECT = ["sentence_count"]  # never pruned, so the headline structural feature stays in the model
cols = list(ALL)
dropped = []
while True:
    v = vif_series(df[cols])
    candidates = v[~v.index.isin(PROTECT)]
    if candidates.iloc[0] <= 10 or len(cols) <= 3:
        break
    dropped.append((candidates.index[0], round(float(candidates.iloc[0]), 1)))
    cols.remove(candidates.index[0])
REDUCED = cols
print("Dropped for VIF > 10 (feature, VIF):", dropped)
print("Reduced feature set:", REDUCED)
print("\nFinal VIFs:\n", vif_series(df[REDUCED]).round(2).to_string())

rec_red, oof_red = cv_eval(make_lr, REDUCED)
print(f"\nLogReg on reduced set: acc {rec_red.acc.mean():.4f} +/- {rec_red.acc.std():.4f}, "
      f"AUC {rec_red.auc.mean():.4f}")

# bootstrap CIs for standardised coefficients
B = 200
rng = np.random.default_rng(SEED)
Xr = df[REDUCED].values
coefs = []
for _ in range(B):
    idx = rng.integers(0, len(Xr), len(Xr))
    coefs.append(make_lr().fit(Xr[idx], y[idx]).named_steps["lr"].coef_[0])
coefs = np.array(coefs)
coef_df = pd.DataFrame({"feature": REDUCED, "coef": coefs.mean(0),
                        "ci_low": np.percentile(coefs, 2.5, axis=0),
                        "ci_high": np.percentile(coefs, 97.5, axis=0)})
coef_df["sign_stable"] = (coef_df.ci_low * coef_df.ci_high) > 0
coef_df = coef_df.sort_values("coef", key=abs, ascending=False)
coef_df.to_csv(os.path.join(OUT_DIR, "lr_bootstrap_coefs.csv"), index=False)
print("\nStandardised LR coefficients (positive = predicts real), 95% bootstrap CI:")
print(coef_df.round(3).to_string(index=False))

# permutation importance on a held-out grouped split
tr, te = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(Xr, y, groups))
m_pi = make_lr().fit(Xr[tr], y[tr])
pi = permutation_importance(m_pi, Xr[te], y[te], scoring="roc_auc", n_repeats=10, random_state=SEED)
pi_df = pd.DataFrame({"feature": REDUCED, "auc_drop": pi.importances_mean, "sd": pi.importances_std}
                     ).sort_values("auc_drop", ascending=False)
pi_df.to_csv(os.path.join(OUT_DIR, "permutation_importance.csv"), index=False)
print("\nPermutation importance (drop in held-out AUC when the feature is shuffled):")
print(pi_df.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
cd = coef_df.sort_values("coef")
ax[0].barh(cd.feature, cd.coef, xerr=[cd.coef - cd.ci_low, cd.ci_high - cd.coef],
           color=["#4C72B0" if c > 0 else "#DD8452" for c in cd.coef])
ax[0].axvline(0, color="black", lw=0.8); ax[0].set_title("LogReg coefficients (95% bootstrap CI)")
pd_ = pi_df.sort_values("auc_drop")
ax[1].barh(pd_.feature, pd_.auc_drop, xerr=pd_["sd"], color="#4C72B0")
ax[1].set_title("Permutation importance (held-out AUC drop)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "lr_coefs_and_permutation_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

# optional: formal Logit significance on the reduced set (only if statsmodels is installed)
try:
    import statsmodels.api as sm
    Xs = sm.add_constant(pd.DataFrame(StandardScaler().fit_transform(df[REDUCED]), columns=REDUCED))
    logit = sm.Logit(y, Xs).fit(disp=0, maxiter=200)
    print("\nstatsmodels Logit converged:", logit.mle_retvals.get("converged"))
    print(logit.summary().tables[1])
except ImportError:
    print("\n(statsmodels not installed; skipped formal Logit. Run: pip install statsmodels)")

## 9. Error analysis with per-class rates

In [ ]:
pred_red = (oof_red >= 0.5).astype(int)
cm = confusion_matrix(y, pred_red, labels=[0, 1])  # rows: true synthetic, real
n_syn_tot, n_real_tot = cm[0].sum(), cm[1].sum()
print("Out-of-fold confusion matrix (rows=true, cols=pred; order synthetic, real):\n", cm)
print(f"Synthetic misclassified as real: {cm[0,1]} / {n_syn_tot} = {cm[0,1]/n_syn_tot:.2%}")
print(f"Real misclassified as synthetic: {cm[1,0]} / {n_real_tot} = {cm[1,0]/n_real_tot:.2%}")

err = df.assign(pred=pred_red, correct=(pred_red == y))
print("\nMean sentence_count by (true class, correct?):")
print(err.groupby(["source", "correct"])["sentence_count"].agg(["mean", "count"]).round(2).to_string())

## 10. Confidence-based triage (supplies the missing numbers for the paper)
Uses out-of-fold probabilities. Confidence = |2p - 1|. Note: LR probabilities are not guaranteed to be
calibrated; say so in the paper or add `CalibratedClassifierCV`.

In [ ]:
conf = np.abs(2 * oof_red - 1)
triage = []
for c in np.round(np.arange(0, 1.0, 0.1), 2):
    keep = conf >= c
    triage.append({"min_confidence": c,
                   "auto_classified": int(keep.sum()),
                   "coverage": keep.mean(),
                   "accuracy_auto": (pred_red[keep] == y[keep]).mean() if keep.any() else np.nan,
                   "flagged_for_review": int((~keep).sum()),
                   "errors_caught_in_flagged": int(((pred_red != y) & ~keep).sum())})
triage = pd.DataFrame(triage)
triage.to_csv(os.path.join(OUT_DIR, "triage_table.csv"), index=False)
print(triage.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(triage.coverage, triage.accuracy_auto, marker="o")
ax.set_xlabel("Coverage (share auto-classified)"); ax.set_ylabel("Accuracy on auto-classified")
ax.set_title("Selective accuracy vs coverage (out-of-fold)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "triage_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

## 11. How to read the results
* If `sentence_count only` or `length only` already gets close to `all features`, the headline accuracy is mostly
  a length/format fingerprint and the paper should say so.
* If `content/pragmatic only` stays well above 50%, there is signal beyond structure. Report that number; it is
  the more interesting finding.
* `first_person_rate` is **prompt-induced**: the paraphrasing prompt (MentalChat16K Table 5) tells Mistral to write
  the coach's response "in the first-person perspective". Do not interpret that gap as a human-authorship effect.
* Features computed on responses of very different length are only comparable via rates and the length-matched d.